In [ ]:
!git clone https://github.com/Mohamedmagdy21/sentiment_analyzer.git

%cd sentiment_analyzer

# Patch: remove tokenizer kwarg from Trainer() (transformers 5.x compat)
import re
with open("training/trainers/hf_trainer.py", "r") as f:
    content = f.read()
content = content.replace(
    "            tokenizer=self.tokenizer\n",
    ""
)
with open("training/trainers/hf_trainer.py", "w") as f:
    f.write(content)

# Add T4 GPU patch for _get_num_items_in_batch
patch = '''
# Patch transformers 5.x _get_num_items_in_batch for T4 GPU compat
from transformers import trainer as trainer_module
orig_get_num_items = trainer_module.Trainer._get_num_items_in_batch
def patched_get_num_items(self, batch_samples, device):
    try:
        return orig_get_num_items(self, batch_samples, device)
    except Exception:
        return sum(batch["labels"].numel() for batch in batch_samples)
trainer_module.Trainer._get_num_items_in_batch = patched_get_num_items
'''
with open("training/trainers/hf_trainer.py", "a") as f:
    f.write(patch)

print("Patched hf_trainer.py for transformers 5.x + T4 GPU")

!pip install -r requirements.txt
# Reinstall torch+vision with CUDA 12.4 for T4 GPU support
!pip install torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124 --force-reinstall

In [ ]:
import os
os.makedirs("Data/raw/sent", exist_ok=True)
!cp /kaggle/input/sentiment140/training.1600000.processed.noemoticon.csv \
    Data/raw/sent/training.1600000.processed.noemoticon.csv
print("Copied raw Twitter data from Kaggle input.")

In [ ]:
!python -m preprocessing.preprocess dataset=twitter hydra.run.dir=.

In [ ]:
!HYDRA_FULL_ERROR=1 python -m training.train \
    dataset=twitter \
    model=twitter_roberta \
    hydra.run.dir=.